In [ ]:
from src.data_ingestion.data_loader import IngestionFactory,DataLoader
from src.data_ingestion.data_validator import ValidationFactory,DataValidator
from src.data_ingestion.data_preprocessor import PreprocessingFactory
from src.features.customer_features import CustomerFeatureExtractor
from src.features.product_features import ProductFeatureExtractor
from src.features.training_data_builder import TrainingDataBuilder
from src.training.data_splitter import TemporalDataSplitter
from src.models.baseline_models import PopularityRecommender,PersonalFrequencyRecommender
from src.evaluation.metrics import RankingMetrics
from src.models.lightgbm_ranker import LightGBMRanker
from collections import Counter

In [ ]:
#Load CSV
ingestion = IngestionFactory.create(
    source_type="csv",
    file_path = 'data/raw/data_raw.csv',
    date_columns=["order_date", "first_order_date", "last_order_date"]
)

loader = DataLoader(ingestion)
raw = loader.load()
df = raw.transactions

In [ ]:
#Validation - only proceed when is valid = True
validator = DataValidator(
    rules=ValidationFactory.default_rules(),   # Using ValidationFactory 
    strict_mode=False                          # False = allow WARNING, fail only ERROR/CRITICAL
)
report = validator.validate(df)
report.is_valid

In [ ]:
#Preprocessing
preprocessor = PreprocessingFactory.create(
    method="sequence",
    min_orders=2
)
prepared = preprocessor.transform(df)


In [ ]:
#Customer Feature Extration
cust_ext = CustomerFeatureExtractor()
customer_profiles = cust_ext.extract(prepared)

In [ ]:
#Producgt Feature Extration
prod_ext = ProductFeatureExtractor()
product_features = prod_ext.extract(prepared)

In [ ]:
#Build Training Data
builder = TrainingDataBuilder(negative_ratio=5)
training_data = builder.build(
prepared_data=prepared,
product_features=product_features,
customer_profiles=customer_profiles
)

In [ ]:
#Train Test Split
#Issue:CustomerID in train set might not in test set
splitter = TemporalDataSplitter(test_ratio=0.2)
split = splitter.split(training_data)

In [ ]:
train_df = split.train_df
test_df = split.test_df
feature_names = split.feature_names


In [ ]:
#base model and baseline creation
pop_model = PopularityRecommender().fit(train_df, feature_names)
pf_model = PersonalFrequencyRecommender(smoothing=0.3).fit(train_df, feature_names)


In [ ]:
#
test_scores_pop = pop_model.predict_df(test_df)
test_scores_pf = pf_model.predict_df(test_df)

In [ ]:
evaluator = RankingMetrics(k_values=[1, 3, 5])

result_pop = evaluator.evaluate(pop_model, test_df, feature_names)
print(result_pop.metrics)

In [ ]:
#LGBM Ranker


# Create the model
lgbm_model = LightGBMRanker(
    num_leaves=31,
    learning_rate=0.05,
    num_boost_round=300,
    early_stopping_rounds=30
)

# Fit the model (train_df must contain label, customer_id, order_idx)
lgbm_model.fit(
    train_df=train_df,
    feature_names=feature_names
)
